# Intelligent RAG-Based College Knowledge Chatbot

**Final project notebook**

This notebook documents the implemented Retrieval-Augmented Generation (RAG) workflow for answering college-related questions from a structured knowledge base.

**Knowledge categories:** Attendance, Library, Admissions, Placements, Examinations, and Fees.

**Core technologies:** Python, LangChain, HuggingFace sentence-transformer embeddings, ChromaDB, Groq LLM, Streamlit, and RAGAS.

> **Reproducibility note:** The knowledge-base records are included directly in this notebook so the submission does not depend on a separate CSV file. A Groq API key is requested securely at runtime and is never written into the notebook. The Streamlit interface is represented by the same RAG functions and chat workflow here; Streamlit itself is run as a separate application, not inside a notebook cell.


## 1. Problem domain and objectives

College information is often spread across policies and departmental documents. Students may need to search several sources or contact staff to answer routine questions. This project provides a conversational assistant that retrieves relevant college information and generates concise answers grounded in the retrieved content.

**Objectives**
- Ingest structured college knowledge records.
- Split source text into manageable chunks.
- Generate semantic embeddings.
- Store and retrieve chunks with a persistent vector database.
- Generate grounded answers with a Groq-hosted language model.
- Interpret follow-up questions using conversation history.
- Evaluate answer faithfulness and relevancy with RAGAS.


## 2. Install and import dependencies

In [ ]:
# %pip install -q pandas python-dotenv langchain langchain-community langchain-core langchain-text-splitters langchain-huggingface langchain-chroma langchain-groq sentence-transformers ragas datasets

: 

In [1]:
import os
import hashlib
import pandas as pd
from getpass import getpass
from IPython.display import display

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from dotenv import load_dotenv

print("Core libraries imported successfully.")

Core libraries imported successfully.


## 3. College knowledge base

The following records reproduce the contents of the project's `college_knowledge_base.csv` file. They are embedded here so the notebook can be submitted on its own.

In [2]:
data = [{'category': 'Attendance', 'content': 'Students must maintain a minimum attendance of 80% in each subject to be eligible for semester examinations.', 'source': 'attendance_policy'}, 
        {'category': 'Attendance', 'content': 'Students with medical emergencies must submit valid medical certificates to the academic department.', 'source': 'attendance_policy'}, 
        {'category': 'Library', 'content': 'Students can borrow a maximum of 4 books at a time using their college identity card.', 'source': 'library_policy'}, 
        {'category': 'Library', 'content': 'The college library operates from 9 AM to 5 PM, Monday through Saturday.', 'source': 'library_policy'}, 
        {'category': 'Library', 'content': 'Books must be returned within 14 days. Late returns may result in a fine.', 'source': 'library_policy'}, 
        {'category': 'Admissions', 'content': 'Applicants must submit academic transcripts, identity proof, and completed application forms during admission.', 'source': 'admission_policy'}, 
        {'category': 'Admissions', 'content': 'Admission confirmation is provided after document verification and payment of the applicable admission fee.', 'source': 'admission_policy'}, 
        {'category': 'Placements', 'content': 'Students must maintain a minimum CGPA of 7.0 to participate in campus placement activities.', 'source': 'placement_policy'}, 
        {'category': 'Placements', 'content': 'The placement department organizes aptitude training, technical workshops, and mock interviews.', 'source': 'placement_policy'}, 
        {'category': 'Examinations', 'content': 'Students must carry their college identity cards and examination hall tickets to enter the examination hall.', 'source': 'exam_policy'}, 
        {'category': 'Examinations', 'content': 'Students must arrive at the examination hall at least 15 minutes before the scheduled examination time.', 'source': 'exam_policy'}, 
        {'category': 'Fees', 'content': 'Students can pay their semester fees through the college payment portal before the announced deadline.', 'source': 'fee_policy'}]

df = pd.DataFrame(data)
print("Dataset shape:", df.shape)
display(df)

Dataset shape: (12, 3)


,category,content,source
0,Attendance,Students must maintain a minimum attendance of...,attendance_policy
1,Attendance,Students with medical emergencies must submit ...,attendance_policy
2,Library,Students can borrow a maximum of 4 books at a ...,library_policy
3,Library,The college library operates from 9 AM to 5 PM...,library_policy
4,Library,Books must be returned within 14 days. Late re...,library_policy
5,Admissions,"Applicants must submit academic transcripts, i...",admission_policy
6,Admissions,Admission confirmation is provided after docum...,admission_policy
7,Placements,Students must maintain a minimum CGPA of 7.0 t...,placement_policy
8,Placements,The placement department organizes aptitude tr...,placement_policy
9,Examinations,Students must carry their college identity car...,exam_policy


## 4. Convert records into LangChain documents

Each CSV row becomes one document. Metadata preserves the category and source label for traceability.

In [3]:
documents = [
    Document(
        page_content=f"Category: {row.category}\nInformation: {row.content}",
        metadata={"category": row.category, "source": row.source, "record_id": int(i)}
    )
    for i, row in df.iterrows()
]

print("Documents prepared:", len(documents))
print("Example document:")
print(documents[0].page_content)
print("Metadata:", documents[0].metadata)

Documents prepared: 12
Example document:
Category: Attendance
Information: Students must maintain a minimum attendance of 80% in each subject to be eligible for semester examinations.
Metadata: {'category': 'Attendance', 'source': 'attendance_policy', 'record_id': 0}


## 5. Text chunking

The implementation uses `RecursiveCharacterTextSplitter` with an 800-character chunk size and 120-character overlap.

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120
)

chunks = text_splitter.split_documents(documents)

print("Total documents:", len(documents))
print("Total chunks:", len(chunks))
print("\nFirst chunk:")
print(chunks[0].page_content)

Total documents: 12
Total chunks: 12

First chunk:
Category: Attendance
Information: Students must maintain a minimum attendance of 80% in each subject to be eligible for semester examinations.


## 6. Generate embeddings

The project uses the `sentence-transformers/all-MiniLM-L6-v2` embedding model.

In [5]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

sample_vector = embedding_model.embed_query(
    "What is the minimum attendance required for examinations?"
)

print("Embedding dimensions:", len(sample_vector))
print("First 10 values:", sample_vector[:10])

/opt/anaconda3/envs/tf311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1114 14:55:40.040000 13454 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8131.25it/s]


Embedding dimensions: 384
First 10 values: [0.0998721644282341, 0.04587428271770477, -0.03254128247499466, -0.014470677822828293, -0.13457556068897247, 0.018087022006511688, -0.06862696260213852, 0.0199266504496336, -0.02160484530031681, 0.011700693517923355]


## 7. Create a persistent ChromaDB vector store

A stable collection name is generated from the knowledge-base records. The database is stored in `./chroma_db` relative to this notebook.

In [6]:
knowledge_base_text = "\n".join(
    f"{row.category}|{row.content}|{row.source}"
    for row in df.itertuples(index=False)
)

collection_hash = hashlib.sha256(
    knowledge_base_text.encode("utf-8")
).hexdigest()

collection_name = "college_kb_" + collection_hash[:20]

# Remove the existing version of this collection before rebuilding it.
# This prevents duplicate records when the cell is run repeatedly.
existing_db = Chroma(
    collection_name=collection_name,
    embedding_function=embedding_model,
    persist_directory="./chroma_db"
)

existing_db.delete_collection()

# Create a fresh collection using the current document chunks.
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name=collection_name,
    persist_directory="./chroma_db"
)

print("ChromaDB collection:", collection_name)
print("Stored vectors:", vector_db._collection.count())


ChromaDB collection: college_kb_1a6b95a0cd7b006d6f03
Stored vectors: 12


## 8. Test semantic retrieval

The retriever returns the most relevant chunks for a student's question.

In [7]:
query = "What is the minimum attendance needed to sit for semester examinations?"

retrieved_docs = vector_db.similarity_search(query, k=3)

for index, doc in enumerate(retrieved_docs, start=1):
    print(f"Result {index}")
    print("Category:", doc.metadata.get("category"))
    print("Source:", doc.metadata.get("source"))
    print("Content:", doc.page_content)
    print("-" * 60)

Result 1
Category: Attendance
Source: attendance_policy
Content: Category: Attendance
Information: Students must maintain a minimum attendance of 80% in each subject to be eligible for semester examinations.
------------------------------------------------------------
Result 2
Category: Examinations
Source: exam_policy
Content: Category: Examinations
Information: Students must arrive at the examination hall at least 15 minutes before the scheduled examination time.
------------------------------------------------------------
Result 3
Category: Placements
Source: placement_policy
Content: Category: Placements
Information: Students must maintain a minimum CGPA of 7.0 to participate in campus placement activities.
------------------------------------------------------------


## 9. Configure Groq LLM

Provided the Groq API key when prompted. The key is kept in the current kernel session and is not saved in this notebook.

In [8]:
load_dotenv()

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

print("Groq model configured.")

Groq model configured.


## 10. RAG prompt and answer generation

The model answers from retrieved context. If the context does not support an answer, it must return the fallback message.

In [9]:
rag_prompt = PromptTemplate(
    input_variables=["context", "question", "history"],
    template="""
You are a helpful college knowledge assistant.

Answer the user's question using the provided context
and conversation history.

Rules:
1. Use the context as the primary source of information.
2. Use conversation history to understand follow-up questions.
3. You may make simple, direct logical conclusions from the context.
4. Do not invent additional penalties, rules, or consequences.
5. If the answer cannot reasonably be determined from the context,
   say exactly:
   "Sorry, I couldn't find this information in the uploaded knowledge base."
6. Answer clearly and concisely.

Conversation History:
{history}

Context:
{context}

Question:
{question}

Answer:
"""
)


def retrieve_context(question, k=5, threshold=0.50):
    scored = vector_db.similarity_search_with_score(question, k=k)

    relevant = []
    for doc, distance in scored:
        # Convert ChromaDB distance into a bounded relevance score.
        score = 1.0 / (1.0 + float(distance))

        if score >= threshold:
            relevant.append((doc, score))

    context = "\n\n".join(doc.page_content for doc, _ in relevant)

    return relevant, context


def rag_answer(question, history=""):
    relevant, context = retrieve_context(question)
    if not relevant:
        return "Sorry, I couldn't find this information in the uploaded knowledge base.", relevant
    final_prompt = rag_prompt.format(
        context=context,
        question=question,
        history=history
    )
    response = llm.invoke(final_prompt)
    return response.content.strip(), relevant

question = "How many books can a student borrow from the library?"
answer, sources = rag_answer(question)

print("Question:", question)
print("Answer:", answer)
print("\nRetrieved sources:")
for doc, score in sources:
    print(f"- {doc.metadata.get('source')} | relevance={score:.3f}")

Question: How many books can a student borrow from the library?
Answer: A student can borrow up to **4 books** at a time.

Retrieved sources:
- library_policy | relevance=0.691


## 11. Conversational follow-up question rewriting

A short follow-up such as “What if I can't?” is rewritten as a standalone question using previous turns.

In [10]:
rewrite_prompt = PromptTemplate(
    input_variables=["history", "question"],
    template="""
You rewrite the latest user question as a standalone question.

Resolve references such as "it", "that", "they", and short fragments
using the conversation history. Preserve the user's meaning.
Do not answer the question. Return only the rewritten question.

Conversation History:
{history}

Latest Question:
{question}

Standalone Question:
"""
)

def rewrite_follow_up(question, history):
    formatted = rewrite_prompt.format(history=history, question=question)
    rewritten = llm.invoke(formatted).content.strip()
    return rewritten if rewritten else question

demo_history = (
    "user: What is the minimum attendance required?\n"
    "assistant: The minimum attendance required is 80%."
)
follow_up = "What if I can't?"
standalone_question = rewrite_follow_up(follow_up, demo_history)

print("Original follow-up:", follow_up)
print("Rewritten question:", standalone_question)

Original follow-up: What if I can't?
Rewritten question: What if I can't meet the minimum attendance requirement?


## 12. RAGAS evaluation dataset

Ten evaluation questions cover the six knowledge categories. Reference answers are derived from the included college knowledge records. The evaluation-question set is kept separate from the knowledge base to avoid test-answer leakage.

In [11]:
test_cases = [{'question': 'What is the minimum attendance required in each subject to be eligible for semester examinations?', 
               'reference': 'Students must maintain a minimum attendance of 80% in each subject to be eligible for semester examinations.'}, 
              
              {'question': 'What should students submit if they have a medical emergency affecting attendance?', 
               'reference': 'Students with medical emergencies must submit valid medical certificates to the academic department.'}, 
              
              {'question': 'How many books can a student borrow from the college library at one time?', 
               'reference': 'Students can borrow a maximum of 4 books at a time using their college identity card.'}, 
              
              {'question': "What are the college library's operating hours and days?", 
               'reference': 'The college library operates from 9 AM to 5 PM, Monday through Saturday.'},
              
              {'question': 'Within how many days must library books be returned, and what may happen if they are returned late?', 
               'reference': 'Books must be returned within 14 days. Late returns may result in a fine.'}, 
              
              {'question': 'Which documents must applicants submit during admission?', 
               'reference': 'Applicants must submit academic transcripts, identity proof, and completed application forms during admission.'}, 
              
              {'question': 'When is admission confirmation provided?', 
               'reference': 'Admission confirmation is provided after document verification and payment of the applicable admission fee.'}, 
              
              {'question': 'What minimum CGPA is required to participate in campus placement activities?', 
               'reference': 'Students must maintain a minimum CGPA of 7.0 to participate in campus placement activities.'}, 
              
              {'question': 'What activities does the placement department organize?', 
               'reference': 'The placement department organizes aptitude training, technical workshops, and mock interviews.'},
              
              {'question': 'What must students carry to enter the examination hall, and how early must they arrive?', 
               'reference': 'Students must carry their college identity cards and examination hall tickets and arrive at least 15 minutes before the scheduled examination time.'}]

print("Evaluation questions:", len(test_cases))
display(pd.DataFrame(test_cases))

Evaluation questions: 10


,question,reference
0,What is the minimum attendance required in eac...,Students must maintain a minimum attendance of...
1,What should students submit if they have a med...,Students with medical emergencies must submit ...
2,How many books can a student borrow from the c...,Students can borrow a maximum of 4 books at a ...
3,What are the college library's operating hours...,The college library operates from 9 AM to 5 PM...
4,Within how many days must library books be ret...,Books must be returned within 14 days. Late re...
5,Which documents must applicants submit during ...,"Applicants must submit academic transcripts, i..."
6,When is admission confirmation provided?,Admission confirmation is provided after docum...
7,What minimum CGPA is required to participate i...,Students must maintain a minimum CGPA of 7.0 t...
8,What activities does the placement department ...,The placement department organizes aptitude tr...
9,What must students carry to enter the examinat...,Students must carry their college identity car...


## 13. Run the RAG pipeline for all evaluation questions

This cell collects the generated response and the retrieved context for each question. RAGAS needs all three: user input, response, and retrieved contexts; reference answers are also supplied.

In [12]:
evaluation_rows = []

for index, item in enumerate(test_cases, start=1):
    question = item["question"]
    relevant, context = retrieve_context(question)

    if relevant:
        prompt_text = rag_prompt.format(
            context=context,
            question=question,
            history=""
        )
        generated_answer = llm.invoke(prompt_text).content.strip()
        retrieved_contexts = [doc.page_content for doc, score in relevant]
    else:
        generated_answer = (
            "Sorry, I couldn't find this information in the uploaded knowledge base."
        )
        retrieved_contexts = []

    evaluation_rows.append({
        "user_input": question,
        "retrieved_contexts": retrieved_contexts,
        "response": generated_answer,
        "reference": item["reference"]
    })

    print(f"Completed {index}/10")
    print("Question:", question)
    print("Answer:", generated_answer)
    print("-" * 70)

evaluation_df = pd.DataFrame([
    {
        "question": row["user_input"],
        "answer": row["response"],
        "reference": row["reference"],
        "retrieved_context_count": len(row["retrieved_contexts"])
    }
    for row in evaluation_rows
])
display(evaluation_df)

Completed 1/10
Question: What is the minimum attendance required in each subject to be eligible for semester examinations?
Answer: The minimum attendance required in each subject is **80%**.
----------------------------------------------------------------------
Completed 2/10
Question: What should students submit if they have a medical emergency affecting attendance?
Answer: They should submit a valid medical certificate to the academic department.
----------------------------------------------------------------------
Completed 3/10
Question: How many books can a student borrow from the college library at one time?
Answer: 4 books.
----------------------------------------------------------------------
Completed 4/10
Question: What are the college library's operating hours and days?
Answer: The college library is open Monday through Saturday, from 9 AM to 5 PM.
----------------------------------------------------------------------
Completed 5/10
Question: Within how many days must libra

,question,answer,reference,retrieved_context_count
0,What is the minimum attendance required in eac...,The minimum attendance required in each subjec...,Students must maintain a minimum attendance of...,4
1,What should students submit if they have a med...,They should submit a valid medical certificate...,Students with medical emergencies must submit ...,1
2,How many books can a student borrow from the c...,4 books.,Students can borrow a maximum of 4 books at a ...,1
3,What are the college library's operating hours...,The college library is open Monday through Sat...,The college library operates from 9 AM to 5 PM...,1
4,Within how many days must library books be ret...,Library books must be returned within 14 days....,Books must be returned within 14 days. Late re...,1
5,Which documents must applicants submit during ...,"Applicants must submit academic transcripts, i...","Applicants must submit academic transcripts, i...",3
6,When is admission confirmation provided?,Admission confirmation is provided after the d...,Admission confirmation is provided after docum...,2
7,What minimum CGPA is required to participate i...,The minimum CGPA required is **7.0**.,Students must maintain a minimum CGPA of 7.0 t...,2
8,What activities does the placement department ...,The placement department organizes aptitude tr...,The placement department organizes aptitude tr...,1
9,What must students carry to enter the examinat...,Students must bring their college identity car...,Students must carry their college identity car...,2


## 14. RAGAS metrics: faithfulness and answer relevancy

Faithfulness measures whether the answer is supported by retrieved context. Answer relevancy measures whether the answer addresses the user's question. The following cell uses the RAGAS 0.4.3 compatibility interface used for this project's evaluation.

In [13]:
from ragas import EvaluationDataset, evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import Faithfulness, ResponseRelevancy

ragas_dataset = EvaluationDataset.from_list(evaluation_rows)

evaluator_llm = LangchainLLMWrapper(llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(embedding_model)

metrics = [
    Faithfulness(llm=evaluator_llm),
    ResponseRelevancy(
        llm=evaluator_llm,
        embeddings=evaluator_embeddings,
        strictness=1
    )
]

ragas_result = evaluate(
    dataset=ragas_dataset,
    metrics=metrics,
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    raise_exceptions=True
)

ragas_results_df = ragas_result.to_pandas()
display(ragas_results_df)

ragas_summary = ragas_results_df[
    ["faithfulness", "answer_relevancy"]
].mean(numeric_only=True).to_frame("average_score")

display(ragas_summary)

/var/folders/rs/r90vs3p52xj_8lslpq79wy_m0000gn/T/ipykernel_13454/595575218.py:4: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, ResponseRelevancy
/var/folders/rs/r90vs3p52xj_8lslpq79wy_m0000gn/T/ipykernel_13454/595575218.py:4: DeprecationWarning: Importing ResponseRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ResponseRelevancy
  from ragas.metrics import Faithfulness, ResponseRelevancy
/var/folders/rs/r90vs3p52xj_8lslpq79wy_m0000gn/T/ipykernel_13454/595575218.py:8: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_fact

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy
0,What is the minimum attendance required in eac...,[Category: Attendance\nInformation: Students m...,The minimum attendance required in each subjec...,Students must maintain a minimum attendance of...,1.0,0.859684
1,What should students submit if they have a med...,[Category: Attendance\nInformation: Students w...,They should submit a valid medical certificate...,Students with medical emergencies must submit ...,0.0,0.678758
2,How many books can a student borrow from the c...,[Category: Library\nInformation: Students can ...,4 books.,Students can borrow a maximum of 4 books at a ...,1.0,0.607206
3,What are the college library's operating hours...,[Category: Library\nInformation: The college l...,The college library is open Monday through Sat...,The college library operates from 9 AM to 5 PM...,1.0,0.890445
4,Within how many days must library books be ret...,[Category: Library\nInformation: Books must be...,Library books must be returned within 14 days....,Books must be returned within 14 days. Late re...,1.0,0.755574
5,Which documents must applicants submit during ...,[Category: Admissions\nInformation: Applicants...,"Applicants must submit academic transcripts, i...","Applicants must submit academic transcripts, i...",1.0,0.966891
6,When is admission confirmation provided?,[Category: Admissions\nInformation: Admission ...,Admission confirmation is provided after the d...,Admission confirmation is provided after docum...,1.0,1.000000
7,What minimum CGPA is required to participate i...,[Category: Placements\nInformation: Students m...,The minimum CGPA required is **7.0**.,Students must maintain a minimum CGPA of 7.0 t...,1.0,0.815296
8,What activities does the placement department ...,[Category: Placements\nInformation: The placem...,The placement department organizes aptitude tr...,The placement department organizes aptitude tr...,1.0,1.000000
9,What must students carry to enter the examinat...,[Category: Examinations\nInformation: Students...,Students must bring their college identity car...,Students must carry their college identity car...,1.0,0.802335


,average_score
faithfulness,0.900000
answer_relevancy,0.837619


## 15. Recorded evaluation results

The following scores are the actual output reported after running `evaluate_rag.py` for this project. They are included here as the completed evaluation result.

In [15]:
recorded_results = pd.DataFrame(
    {
        "Metric": ["Faithfulness", "Answer relevancy"],
        "Average score": [0.900000, 0.837619]
    }
)
display(recorded_results)

print("Faithfulness: 0.900000")
print("Answer relevancy: 0.837619")

,Metric,Average score
0,Faithfulness,0.900000
1,Answer relevancy,0.837619


Faithfulness: 0.900000
Answer relevancy: 0.837619


## 16. Results interpretation

| RAGAS metric | Score | Interpretation |
|---|---:|---|
| Faithfulness | 0.900000 | Generated answers were strongly supported by retrieved context in this evaluation. |
| Answer relevancy | 0.837619 | Generated answers were generally relevant to the questions, with room for improvement. |

These are average scores from the 10-question evaluation run. They are not a guarantee that every possible answer is correct. The evaluation set is small and based on the current college knowledge base.


## 17. Conclusion

The project implements a document-grounded college knowledge assistant using retrieval-augmented generation. Structured college information is converted into documents, split into chunks, embedded, and stored in ChromaDB. At query time, semantic retrieval supplies relevant context to a Groq-hosted language model, which generates a grounded response. Conversation history supports follow-up questions.

The initial RAGAS evaluation produced a faithfulness score of **0.90** and an answer relevancy score of **0.837619** across ten test questions. These results provide an initial quality measurement and a baseline for future improvements.

**Limitations:** The knowledge base contains 12 records across six categories. The evaluation covers ten questions and should be expanded with more varied, paraphrased, and multi-document questions for a more robust assessment.
